<a href="https://colab.research.google.com/github/Rachana826/Dental-Clinic-project/blob/main/Dental_Clinic_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from dataclasses import dataclass
from typing import List, Optional

OPEN_HOUR = 9
CLOSE_HOUR = 17


@dataclass
class Appointment:
    patient_name: str
    duration_hours: int
    priority: int = 5

    room_id: Optional[int] = None
    dentist_id: Optional[int] = None
    start_time: Optional[int] = None

    def __post_init__(self):
        if not (1 <= self.duration_hours <= 8):
            raise ValueError(f"Invalid duration for {self.patient_name}: must be between 1 and 8 hours.")

    @property
    def end_time(self) -> Optional[int]:
        if self.start_time is None:
            return None
        return self.start_time + self.duration_hours

    def copy(self) -> "Appointment":
        return Appointment(
            patient_name=self.patient_name,
            duration_hours=self.duration_hours,
            priority=self.priority,
        )


class ClinicScheduler:
    def __init__(self, num_rooms: int, num_dentists: int):
        if not (1 <= num_rooms <= 10) or not (1 <= num_dentists <= 10):
            raise ValueError("Rooms and dentists must both be between 1 and 10.")

        self.num_rooms = num_rooms
        self.num_dentists = num_dentists

        self.room_free_at = [OPEN_HOUR] * num_rooms
        self.dentist_free_at = [OPEN_HOUR] * num_dentists

        self.scheduled: List[Appointment] = []
        self.unscheduled: List[Appointment] = []

    def schedule(self, appointments: List[Appointment]) -> None:

        queue = sorted(appointments, key=lambda a: (a.priority, -a.duration_hours))

        for appt in queue:
            best_start = CLOSE_HOUR + 1
            selected_room = -1
            selected_dentist = -1

            for r_idx in range(self.num_rooms):
                for d_idx in range(self.num_dentists):
                    possible_start = max(self.room_free_at[r_idx], self.dentist_free_at[d_idx])

                    if possible_start < best_start:
                        best_start = possible_start
                        selected_room = r_idx
                        selected_dentist = d_idx

            if best_start + appt.duration_hours <= CLOSE_HOUR:
                appt.start_time = best_start
                appt.room_id = selected_room + 1
                appt.dentist_id = selected_dentist + 1

                finish_time = best_start + appt.duration_hours
                self.room_free_at[selected_room] = finish_time
                self.dentist_free_at[selected_dentist] = finish_time

                self.scheduled.append(appt)
            else:
                self.unscheduled.append(appt)

    def print_schedule(self) -> None:
        print(f"\n--- Clinic Schedule ({self.num_rooms} Rooms, {self.num_dentists} Dentists) ---")

        for r_id in range(1, self.num_rooms + 1):
            print(f"\nOperatory {r_id}:")
            room_appts = sorted(
                [a for a in self.scheduled if a.room_id == r_id],
                key=lambda x: x.start_time
            )

            if not room_appts:
                print("  (No appointments booked)")
                continue

            for appt in room_appts:
                print(f"  {appt.start_time:02d}:00 - {appt.end_time:02d}:00 | "
                      f"{appt.patient_name:<12} (Dr. #{appt.dentist_id}, {appt.duration_hours}h)")

        if self.unscheduled:
            print("\nUnscheduled Patients:")
            for appt in self.unscheduled:
                print(f"  - {appt.patient_name} ({appt.duration_hours}h, Priority {appt.priority})")

        last_finish = max((a.end_time for a in self.scheduled), default=OPEN_HOUR)
        print(f"\nDay ends at: {last_finish:02d}:00 ({last_finish - OPEN_HOUR} hours total)")


def analyze_capacity(appointments: List[Appointment], num_dentists: int):
    print("\n--- Operatory Capacity Analysis ---")

    for room_count in range(1, 11):
        test_appts = [a.copy() for a in appointments]
        scheduler = ClinicScheduler(num_rooms=room_count, num_dentists=num_dentists)
        scheduler.schedule(test_appts)

        last_finish = max((a.end_time for a in scheduler.scheduled), default=OPEN_HOUR)
        hours_worked = last_finish - OPEN_HOUR
        leftover = len(scheduler.unscheduled)

        status = "All booked" if leftover == 0 else f"{leftover} left over"
        print(f"{room_count:2d} rooms -> Day finished in {hours_worked}h ({status})")


def get_sample_data() -> List[Appointment]:
    return [
        Appointment("J. Smith", duration_hours=2, priority=1),
        Appointment("A. Rivera", duration_hours=1, priority=2),
        Appointment("M. Chen", duration_hours=3, priority=2),
        Appointment("K. Patel", duration_hours=1, priority=3),
        Appointment("D. Johnson", duration_hours=4, priority=1),
        Appointment("L. Nguyen", duration_hours=2, priority=3),
        Appointment("S. Brown", duration_hours=1, priority=4),
        Appointment("R. Garcia", duration_hours=2, priority=2),
        Appointment("T. Wilson", duration_hours=3, priority=3),
        Appointment("P. Davis", duration_hours=1, priority=1),
        Appointment("E. Martinez", duration_hours=2, priority=4),
        Appointment("N. Clark", duration_hours=1, priority=5),
    ]


if __name__ == "__main__":
    num_rooms = 3
    num_dentists = 2

    clinic = ClinicScheduler(num_rooms, num_dentists)
    clinic.schedule(get_sample_data())
    clinic.print_schedule()

    analyze_capacity(get_sample_data(), num_dentists)


--- Clinic Schedule (3 Rooms, 2 Dentists) ---

Operatory 1:
  09:00 - 13:00 | D. Johnson   (Dr. #1, 4h)
  13:00 - 15:00 | R. Garcia    (Dr. #1, 2h)
  15:00 - 16:00 | A. Rivera    (Dr. #1, 1h)
  16:00 - 17:00 | K. Patel     (Dr. #1, 1h)

Operatory 2:
  09:00 - 11:00 | J. Smith     (Dr. #2, 2h)
  11:00 - 12:00 | P. Davis     (Dr. #2, 1h)
  12:00 - 15:00 | M. Chen      (Dr. #2, 3h)
  15:00 - 17:00 | L. Nguyen    (Dr. #2, 2h)

Operatory 3:
  (No appointments booked)

Unscheduled Patients:
  - T. Wilson (3h, Priority 3)
  - E. Martinez (2h, Priority 4)
  - S. Brown (1h, Priority 4)
  - N. Clark (1h, Priority 5)

Day ends at: 17:00 (8 hours total)

--- Operatory Capacity Analysis ---
 1 rooms -> Day finished in 8h (8 left over)
 2 rooms -> Day finished in 8h (4 left over)
 3 rooms -> Day finished in 8h (4 left over)
 4 rooms -> Day finished in 8h (4 left over)
 5 rooms -> Day finished in 8h (4 left over)
 6 rooms -> Day finished in 8h (4 left over)
 7 rooms -> Day finished in 8h (4 left ove